# Validate LLM Ground Truth Against Human Labels

This notebook is an audit-only validation notebook. It does not create training data, does not modify retrieval runs, and does not expose or write stable qrels.

Its only purpose is to compare completed human relevance labels against the LLM-generated ground truth. It supports partial human annotation files: rows without `human_relevance` are removed before computing agreement metrics.


## 1. Setup

Expected inputs:

- `groundtruth_outputs/annotation/llm_groundtruth_labels.jsonl`
- one human validation file in `groundtruth_outputs/human_validation/`

The human file can be either:

- Excel workbook, usually `human_validation_annotation_sheet_20.xlsx`, using sheet `annotation`
- CSV export, usually `human_validation_annotation_sheet_completed.csv`

The human file must contain `query_id`, `human_relevance`, and either `doc_id` or `_doc_id_internal`. Unannotated rows are ignored.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

import pandas as pd
from IPython.display import display


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
HUMAN_VALIDATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "human_validation"

LLM_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"

# Set this to a specific file if needed. Leave as None to auto-pick the newest human validation file.
HUMAN_LABELS_PATH_OVERRIDE = None

RELEVANCE_LABELS = [0, 1, 2, 3]
BINARY_RELEVANCE_THRESHOLD = 1

print("Finalproject root:", FINALPROJECT_ROOT)
print("LLM labels path:", LLM_LABELS_PATH)
print("Human validation dir:", HUMAN_VALIDATION_OUTPUT_DIR)


Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
LLM labels path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\llm_groundtruth_labels.jsonl
Human validation dir: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\human_validation


## 2. Load Labels

This section loads existing files only. It auto-selects the newest human validation file when `HUMAN_LABELS_PATH_OVERRIDE` is `None`, then filters out rows where `human_relevance` is blank.


In [2]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {jsonl_path}:{line_number}") from error
    return records


def find_human_validation_file(human_validation_dir: Path, override_path: Optional[Path] = None) -> Path:
    """Find the human validation file to use for agreement measurement."""
    if override_path is not None:
        selected_path = Path(override_path)
        if not selected_path.is_absolute():
            selected_path = human_validation_dir / selected_path
        if not selected_path.exists():
            raise FileNotFoundError(f"Human validation override file not found: {selected_path}")
        return selected_path

    candidate_paths = []
    candidate_paths.extend(human_validation_dir.glob("human_validation_annotation_sheet_completed.csv"))
    candidate_paths.extend(human_validation_dir.glob("human_validation_annotation_sheet_*.xlsx"))
    candidate_paths.extend(human_validation_dir.glob("human_validation_annotation_sheet*.csv"))
    candidate_paths.extend(human_validation_dir.glob("human_validation_annotation_sheet.xlsx"))

    candidate_paths = [path for path in candidate_paths if path.exists() and not path.name.startswith("~$")]
    if not candidate_paths:
        raise FileNotFoundError(f"No human validation file found in {human_validation_dir}")

    return sorted(candidate_paths, key=lambda path: path.stat().st_mtime, reverse=True)[0]


def load_llm_labels(llm_labels_path: Path) -> pd.DataFrame:
    """Load LLM labels as one row per query-document pair."""
    if not llm_labels_path.exists():
        raise FileNotFoundError(f"LLM labels file not found: {llm_labels_path}")

    label_dataframe = pd.DataFrame(load_jsonl_records(llm_labels_path))
    required_columns = {"query_id", "doc_id", "relevance"}
    missing_columns = required_columns - set(label_dataframe.columns)
    if missing_columns:
        raise ValueError(f"LLM labels are missing required columns: {sorted(missing_columns)}")

    label_dataframe = label_dataframe.rename(columns={"relevance": "llm_relevance"}).copy()
    label_dataframe["query_id"] = label_dataframe["query_id"].astype(int)
    label_dataframe["doc_id"] = label_dataframe["doc_id"].astype(int)
    label_dataframe["llm_relevance"] = label_dataframe["llm_relevance"].astype(int)

    duplicate_count = label_dataframe.duplicated(subset=["query_id", "doc_id"]).sum()
    if duplicate_count:
        raise ValueError(f"LLM labels contain {duplicate_count} duplicate query-doc pairs.")

    return label_dataframe


def read_human_validation_table(human_labels_path: Path) -> pd.DataFrame:
    """Read the annotation table from an Excel or CSV human validation file."""
    suffix = human_labels_path.suffix.lower()
    if suffix in {".xlsx", ".xlsm", ".xls"}:
        return pd.read_excel(human_labels_path, sheet_name="annotation")
    if suffix == ".csv":
        return pd.read_csv(human_labels_path)
    raise ValueError(f"Unsupported human validation file type: {human_labels_path}")


def load_human_completed_labels(human_labels_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load human labels and return completed rows plus the raw annotation table."""
    if not human_labels_path.exists():
        raise FileNotFoundError(f"Human validation file not found: {human_labels_path}")

    raw_human_dataframe = read_human_validation_table(human_labels_path)
    human_dataframe = raw_human_dataframe.copy()

    if "doc_id" not in human_dataframe.columns and "_doc_id_internal" in human_dataframe.columns:
        human_dataframe = human_dataframe.rename(columns={"_doc_id_internal": "doc_id"})

    required_columns = {"query_id", "doc_id", "human_relevance"}
    missing_columns = required_columns - set(human_dataframe.columns)
    if missing_columns:
        raise ValueError(f"Human labels are missing required columns: {sorted(missing_columns)}")

    human_dataframe["human_relevance"] = pd.to_numeric(human_dataframe["human_relevance"], errors="coerce")
    completed_human_dataframe = human_dataframe.dropna(subset=["human_relevance"]).copy()

    completed_human_dataframe["query_id"] = completed_human_dataframe["query_id"].astype(int)
    completed_human_dataframe["doc_id"] = completed_human_dataframe["doc_id"].astype(int)
    completed_human_dataframe["human_relevance"] = completed_human_dataframe["human_relevance"].astype(int)

    invalid_labels = sorted(set(completed_human_dataframe["human_relevance"]) - set(RELEVANCE_LABELS))
    if invalid_labels:
        raise ValueError(f"Human labels contain invalid relevance values: {invalid_labels}")

    duplicate_count = completed_human_dataframe.duplicated(subset=["query_id", "doc_id"]).sum()
    if duplicate_count:
        raise ValueError(f"Human labels contain {duplicate_count} duplicate query-doc pairs.")

    return completed_human_dataframe, raw_human_dataframe


def build_validation_comparison(
    llm_label_dataframe: pd.DataFrame,
    human_label_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Merge LLM and human labels on query_id/doc_id."""
    comparison_dataframe = human_label_dataframe.merge(
        llm_label_dataframe[["query_id", "doc_id", "llm_relevance"]],
        on=["query_id", "doc_id"],
        how="inner",
    )

    if comparison_dataframe.empty:
        raise ValueError("No overlapping query-doc pairs between human labels and LLM labels.")

    missing_llm_count = len(human_label_dataframe) - len(comparison_dataframe)
    if missing_llm_count:
        print(f"Warning: {missing_llm_count} human-labeled rows do not have matching LLM labels.")

    return comparison_dataframe


HUMAN_LABELS_PATH = find_human_validation_file(
    HUMAN_VALIDATION_OUTPUT_DIR,
    Path(HUMAN_LABELS_PATH_OVERRIDE) if HUMAN_LABELS_PATH_OVERRIDE else None,
)

llm_label_dataframe = load_llm_labels(LLM_LABELS_PATH)
human_label_dataframe, raw_human_annotation_dataframe = load_human_completed_labels(HUMAN_LABELS_PATH)
comparison_dataframe = build_validation_comparison(llm_label_dataframe, human_label_dataframe)

print("Selected human validation file:", HUMAN_LABELS_PATH)
print("Raw human annotation rows:", len(raw_human_annotation_dataframe))
print("Completed human annotation rows:", len(human_label_dataframe))
print("Dropped unannotated rows:", len(raw_human_annotation_dataframe) - len(human_label_dataframe))
print("Completed human queries:", human_label_dataframe["query_id"].nunique())
print("Compared pairs:", len(comparison_dataframe))
print("Compared queries:", comparison_dataframe["query_id"].nunique())

print("\nHuman label distribution:")
display(
    human_label_dataframe["human_relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("human_relevance")
    .reset_index(name="num_pairs")
)

print("\nLLM label distribution on compared pairs:")
display(
    comparison_dataframe["llm_relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("llm_relevance")
    .reset_index(name="num_pairs")
)


Selected human validation file: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\human_validation\human_validation_annotation_sheet_20.xlsx
Raw human annotation rows: 2000
Completed human annotation rows: 1050
Dropped unannotated rows: 950
Completed human queries: 21
Compared pairs: 1050
Compared queries: 21

Human label distribution:


,human_relevance,num_pairs
0,0,292
1,1,635
2,2,72
3,3,51



LLM label distribution on compared pairs:


,llm_relevance,num_pairs
0,0,562
1,1,289
2,2,149
3,3,50


## 3. Agreement Metrics

The metrics are computed in memory and printed only.

- Exact accuracy checks whether the 0-3 labels match exactly.
- Within-one accuracy allows small disagreements such as `2` vs `3`.
- Binary accuracy collapses labels into non-relevant (`0`) and relevant (`1`, `2`, `3`).
- Quadratic weighted Cohen's kappa measures ordinal agreement and penalizes large disagreements more strongly.


In [3]:
def build_confusion_matrix(
    human_labels: list[int],
    llm_labels: list[int],
    labels: list[int],
) -> pd.DataFrame:
    """Build a confusion matrix with human labels as rows and LLM labels as columns."""
    label_to_index = {label: index for index, label in enumerate(labels)}
    matrix = [[0 for _ in labels] for _ in labels]
    for human_label, llm_label in zip(human_labels, llm_labels):
        matrix[label_to_index[int(human_label)]][label_to_index[int(llm_label)]] += 1
    return pd.DataFrame(
        matrix,
        index=[f"human_{label}" for label in labels],
        columns=[f"llm_{label}" for label in labels],
    )


def quadratic_weighted_kappa(
    human_labels: list[int],
    llm_labels: list[int],
    labels: list[int],
) -> float:
    """Compute quadratic weighted Cohen's kappa without requiring sklearn."""
    if len(human_labels) != len(llm_labels):
        raise ValueError("human_labels and llm_labels must have the same length.")
    if not human_labels:
        return 0.0

    label_count = len(labels)
    label_to_index = {label: index for index, label in enumerate(labels)}

    observed_matrix = [[0.0 for _ in labels] for _ in labels]
    human_histogram = [0.0 for _ in labels]
    llm_histogram = [0.0 for _ in labels]

    for human_label, llm_label in zip(human_labels, llm_labels):
        human_index = label_to_index[int(human_label)]
        llm_index = label_to_index[int(llm_label)]
        observed_matrix[human_index][llm_index] += 1.0
        human_histogram[human_index] += 1.0
        llm_histogram[llm_index] += 1.0

    total_count = float(len(human_labels))
    expected_matrix = [
        [(human_histogram[i] * llm_histogram[j]) / total_count for j in range(label_count)]
        for i in range(label_count)
    ]

    weighted_observed = 0.0
    weighted_expected = 0.0
    max_distance_squared = float((label_count - 1) ** 2)
    for i in range(label_count):
        for j in range(label_count):
            weight = ((i - j) ** 2) / max_distance_squared
            weighted_observed += weight * observed_matrix[i][j]
            weighted_expected += weight * expected_matrix[i][j]

    if weighted_expected == 0:
        return 1.0 if weighted_observed == 0 else 0.0
    return 1.0 - (weighted_observed / weighted_expected)


def compute_agreement_metrics(comparison_dataframe: pd.DataFrame) -> dict:
    """Compute agreement metrics between human labels and LLM labels."""
    if comparison_dataframe.empty:
        raise ValueError("comparison_dataframe is empty. Load completed human labels first.")

    human_labels = comparison_dataframe["human_relevance"].astype(int).tolist()
    llm_labels = comparison_dataframe["llm_relevance"].astype(int).tolist()
    absolute_errors = (comparison_dataframe["human_relevance"] - comparison_dataframe["llm_relevance"]).abs()

    human_binary = (comparison_dataframe["human_relevance"] >= BINARY_RELEVANCE_THRESHOLD).astype(int)
    llm_binary = (comparison_dataframe["llm_relevance"] >= BINARY_RELEVANCE_THRESHOLD).astype(int)

    return {
        "compared_pairs": int(len(comparison_dataframe)),
        "compared_queries": int(comparison_dataframe["query_id"].nunique()),
        "exact_accuracy": float((absolute_errors == 0).mean()),
        "within_one_accuracy": float((absolute_errors <= 1).mean()),
        "binary_accuracy": float((human_binary == llm_binary).mean()),
        "mean_absolute_label_error": float(absolute_errors.mean()),
        "quadratic_weighted_kappa": float(
            quadratic_weighted_kappa(human_labels, llm_labels, labels=RELEVANCE_LABELS)
        ),
        "severe_0_vs_3_disagreement_count": int(
            (
                ((comparison_dataframe["human_relevance"] == 0) & (comparison_dataframe["llm_relevance"] == 3))
                | ((comparison_dataframe["human_relevance"] == 3) & (comparison_dataframe["llm_relevance"] == 0))
            ).sum()
        ),
    }


def print_agreement_report(comparison_dataframe: pd.DataFrame) -> None:
    """Print agreement metrics and confusion matrix."""
    metrics = compute_agreement_metrics(comparison_dataframe)
    print("Agreement metrics")
    for metric_name, metric_value in metrics.items():
        if isinstance(metric_value, float):
            print(f"- {metric_name}: {metric_value:.4f}")
        else:
            print(f"- {metric_name}: {metric_value}")

    print("\nConfusion matrix, rows=human labels, columns=LLM labels")
    display(
        build_confusion_matrix(
            comparison_dataframe["human_relevance"].astype(int).tolist(),
            comparison_dataframe["llm_relevance"].astype(int).tolist(),
            labels=RELEVANCE_LABELS,
        )
    )


print_agreement_report(comparison_dataframe)


Agreement metrics
- compared_pairs: 1050
- compared_queries: 21
- exact_accuracy: 0.5410
- within_one_accuracy: 0.9724
- binary_accuracy: 0.6552
- mean_absolute_label_error: 0.4867
- quadratic_weighted_kappa: 0.5961
- severe_0_vs_3_disagreement_count: 0

Confusion matrix, rows=human labels, columns=LLM labels


,llm_0,llm_1,llm_2,llm_3
human_0,246,34,12,0
human_1,303,237,91,4
human_2,13,18,40,1
human_3,0,0,6,45


## 4. Inspect Largest Disagreements

This table is for human audit only. It helps identify whether disagreements are mostly small boundary cases or severe label inversions.


In [4]:
def show_largest_disagreements(comparison_dataframe: pd.DataFrame, top_n: int = 30) -> pd.DataFrame:
    """Return rows with the largest absolute disagreement."""
    if comparison_dataframe.empty:
        raise ValueError("comparison_dataframe is empty. Load completed human labels first.")

    inspection_dataframe = comparison_dataframe.copy()
    inspection_dataframe["absolute_error"] = (
        inspection_dataframe["human_relevance"] - inspection_dataframe["llm_relevance"]
    ).abs()

    display_columns = [
        "query_id",
        "query_text",
        "candidate_id",
        "doc_id",
        "title",
        "recipe_type",
        "human_relevance",
        "llm_relevance",
        "absolute_error",
        "human_notes",
    ]
    available_columns = [column for column in display_columns if column in inspection_dataframe.columns]

    return (
        inspection_dataframe.sort_values(
            ["absolute_error", "query_id", "doc_id"],
            ascending=[False, True, True],
        )
        .head(top_n)[available_columns]
        .reset_index(drop=True)
    )


display(show_largest_disagreements(comparison_dataframe, top_n=30))


,query_id,query_text,candidate_id,doc_id,title,recipe_type,human_relevance,llm_relevance,absolute_error,human_notes
0,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,D046,4441,Phở bò Hà Nội chuẩn vị nước dùng ngọt thanh đơ...,Món nước,0,2,2,NaN
1,26,Bánh khoai tây nhân phô mai,D038,1885,Bánh mì sandwich nhân khoai lang phô mai thơm ...,Món bánh,2,0,2,NaN
2,26,Bánh khoai tây nhân phô mai,D023,4953,"Súp khoai tây phô mai thơm béo, đầy dinh dưỡng...",Món cháo,2,0,2,NaN
3,26,Bánh khoai tây nhân phô mai,D031,6893,Khoai lang lắc phô mai đơn giản thơm ngon ăn v...,Món chiên,2,0,2,NaN
4,26,Bánh khoai tây nhân phô mai,D035,9031,Cách làm khoai lang nướng phô mai thơm ngon...,Ăn vặt,2,0,2,NaN
5,27,Súp gà,D004,921,Canh gà nấm hương,Món chính,1,3,2,NaN
6,32,Cá lóc nướng riềng sả thơm lừng cay cay ngon h...,D028,6275,"Cá quả (cá lóc) kho riềng đậm đà, thơm ngon",Món kho,2,0,2,NaN
7,54,Tự nấu cháo yến mạch dâu tây tại nhà cho bữa s...,D018,5068,Cháo cá hồi yến mạch cực nhanh bổ dưỡng cho ng...,Món cháo,0,2,2,NaN
8,54,Tự nấu cháo yến mạch dâu tây tại nhà cho bữa s...,D019,5106,Cháo sữa yến mạch đơn giản nhiều dinh dưỡng ch...,Món cháo,0,2,2,NaN
9,61,Gà xào sả ớt cay nồng,D017,5824,"Món gà chọi xào lăn mềm ngon, đậm đà chuẩn vị ...",Món xào,1,3,2,NaN
